# CKP02, DocMind RAG - Intercâmbio

## Disciplina:
  - Prompt Engineering and Artificial Intelligence

## Turma
  - 1CCPI - Ciência da Computação

## Integrantes
  - Kauanne Oliveira - RM574191
  - Nayhely Estela - RM571416

<br>

**Diferenciais implementados:** metadata filtering e reranking com cross-encoder.

## Setup e configuração

Instalação das dependências e configuração central: modelos, endereços e chave.
A chave `OLLAMA_API_KEY` fica nos **Secrets** do Colab e nunca aparece no notebook.

**Decisão sobre os embeddings:** o endpoint de embeddings do Ollama Cloud retornou
`401 unauthorized` com a nossa chave, enquanto o chat com `gemma4:cloud` funcionou (200).
Por orientação do professor, os embeddings são gerados com o mesmo modelo aprovado,
`nomic-embed-text`, servido por um Ollama local no Colab, e o chat continua no Ollama Cloud.

| Etapa | Modelo | Onde roda |
|---|---|---|
| Embeddings | `nomic-embed-text` | Ollama local (Colab) |
| Geração (chat) | `gemma4:cloud`, `temperature=0` | Ollama Cloud |

In [1]:
!nvidia-smi --query-gpu=name --format=csv,noheader
!pip install -q langchain "langchain-community<0.4" langchain-ollama langchain-chroma \
    langchain-text-splitters pymupdf chromadb ragas datasets sentence-transformers

/bin/bash: line 1: nvidia-smi: command not found
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.5 MB/s eta 0:00:00
   ━━━

In [2]:
import os
from pathlib import Path
from google.colab import userdata

""" Modelos exigidos """
MODELO_CHAT = "gemma4:cloud"
MODELO_EMBEDDING = "nomic-embed-text"

""" Endereços: chat no Cloud, embeddings no Ollama local """
URL_CHAT = "https://ollama.com"
URL_EMBEDDING = "http://127.0.0.1:11434"

"""Chave do Cloud (Secret do Colab)"""
OLLAMA_API_KEY = userdata.get("OLLAMA_API_KEY").strip()
os.environ["OLLAMA_API_KEY"] = OLLAMA_API_KEY

PASTA_DOCS = Path("/content/docs")
PASTA_DOCS.mkdir(exist_ok=True)
print("Configuração carregada.")

Configuração carregada.


In [3]:
import subprocess, time, requests

!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get(URL_EMBEDDING, timeout=2)
        break
    except requests.exceptions.RequestException:
        time.sleep(1)

!ollama pull {MODELO_EMBEDDING}

In [4]:
from langchain_ollama import OllamaEmbeddings, ChatOllama

embeddings = OllamaEmbeddings(model=MODELO_EMBEDDING, base_url=URL_EMBEDDING)

llm = ChatOllama(
    model=MODELO_CHAT,
    temperature=0,
    base_url=URL_CHAT,
    client_kwargs={"headers": {"Authorization": f"Bearer {OLLAMA_API_KEY}"}},
)

print("Embeddings (dimensões):", len(embeddings.embed_query("teste")))  # 768
print("Chat:", llm.invoke("Diga apenas: ok").content)

Embeddings (dimensões): 768
Chat: ok


## Base de conhecimento

Seis editais reais, em PDF com texto selecionável, de fontes oficiais.
Cada documento recebe metadados (`instituicao`, `nivel`, `tipo`, `url_origem`),
usados para citar a fonte e para o filtro de metadados.

| Documento | Instituição | Nível | Tipo | Fonte |
|---|---|---|---|---|
| Edital Conjunto 002/2026 | UNIFESP | graduação | mobilidade | [link](https://site.unifesp.br/sri/images/Valquiria%20-%20Mobilidade/Edital%20Conjunto%202026/Edital_Conjunto_2026.pdf) |
| Edital PRRI 01/2026 | UNIFAFIBE | graduação | mobilidade | [link](https://unifafibe.com.br/controlesite/intercambio/anexo_inscricao/1/1765386719_edital_intercambio_01_2026.pdf) |
| Edital PROINTER 20/2026 | UFC | graduação | mobilidade | [link](https://prointer.ufc.br/wp-content/uploads/2026/08/sei-6560983-edital-20-2026-1.pdf) |
| Edital 02/2026 (seleção interna PDSE) | UFF | pós-graduação | PDSE | [link](https://editais.uff.br/sites/default/files/arquivos/Edital_02_PDSE_2026_.pdf) |
| Edital 22/2026 (PDSE) | CAPES | pós-graduação | PDSE | [link](https://www.gov.br/capes/pt-br/centrais-de-conteudo/editais/sei_2849220_edital_n__22_2026.pdf/@@display-file/file) |
| Edital 2336, Internacionalização com Inclusão | USP | graduação | mobilidade | [link]("https://prip.usp.br/wp-content/uploads/sites/1632/2026/08/Edital-2336-%E2%80%93-Internacionalizacao-com-Inclusao.pdf") |

Acesso aos links em: 06 de outubro de 2026.

In [5]:
import requests
from google.colab import files

LINKS_PDF = {
    "unifesp_002_2026.pdf": "https://site.unifesp.br/sri/images/Valquiria%20-%20Mobilidade/Edital%20Conjunto%202026/Edital_Conjunto_2026.pdf",
    "unifafibe_prri_01_2026.pdf": "https://unifafibe.com.br/controlesite/intercambio/anexo_inscricao/1/1765386719_edital_intercambio_01_2026.pdf",
    "ufc_prointer_20_2026.pdf": "https://prointer.ufc.br/wp-content/uploads/2026/08/sei-6560983-edital-20-2026-1.pdf",
    "uff_pdse_02_2026.pdf": "https://editais.uff.br/sites/default/files/arquivos/Edital_02_PDSE_2026_.pdf",
    "capes_22_2026.pdf": "https://www.gov.br/capes/pt-br/centrais-de-conteudo/editais/sei_2849220_edital_n__22_2026.pdf/@@display-file/file",
    "usp_internacionalizacao.pdf": "https://prip.usp.br/wp-content/uploads/sites/1632/2026/08/Edital-2336-%E2%80%93-Internacionalizacao-com-Inclusao.pdf",
}

"""Baixa os PDFs que faltam; se não conseguir, pede upload manual."""
def garantir_documentos():
    faltando = []
    for nome, url in LINKS_PDF.items():
        destino = PASTA_DOCS / nome
        if destino.exists():
            continue
        ok = False
        if url:
            try:
                r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
                if r.ok and r.content[:4] == b"%PDF":
                    destino.write_bytes(r.content)
                    ok = True
            except requests.RequestException:
                pass
        print(("baixado:  " if ok else "FALTANDO: ") + nome)
        if not ok:
            faltando.append(nome)
    if faltando:
        print("\nFaça upload destes arquivos:", faltando)
        for nome, conteudo in files.upload().items():
            (PASTA_DOCS / nome).write_bytes(conteudo)

garantir_documentos()

baixado:  unifesp_002_2026.pdf
baixado:  unifafibe_prri_01_2026.pdf
FALTANDO: ufc_prointer_20_2026.pdf
baixado:  uff_pdse_02_2026.pdf
baixado:  capes_22_2026.pdf
baixado:  usp_internacionalizacao.pdf

Faça upload destes arquivos: ['ufc_prointer_20_2026.pdf']


Saving ufc_prointer_20_2026.pdf to ufc_prointer_20_2026.pdf


In [6]:
import textwrap
import pandas as pd
from IPython.display import display
from langchain_community.document_loaders import PyMuPDFLoader

CATALOGO = {
    "unifesp_002_2026.pdf": {
        "instituicao": "UNIFESP", "nivel": "graduacao", "tipo": "mobilidade",
        "url_origem": "https://site.unifesp.br/sri/images/Valquiria%20-%20Mobilidade/Edital%20Conjunto%202026/Edital_Conjunto_2026.pdf",
    },
    "unifafibe_prri_01_2026.pdf": {
        "instituicao": "UNIFAFIBE", "nivel": "graduacao", "tipo": "mobilidade",
        "url_origem": "https://unifafibe.com.br/controlesite/intercambio/anexo_inscricao/1/1765386719_edital_intercambio_01_2026.pdf",
    },
    "ufc_prointer_20_2026.pdf": {
        "instituicao": "UFC", "nivel": "graduacao", "tipo": "mobilidade",
        "url_origem": "https://prointer.ufc.br/wp-content/uploads/2026/08/sei-6560983-edital-20-2026-1.pdf",
    },
    "uff_pdse_02_2026.pdf": {
        "instituicao": "UFF", "nivel": "pos", "tipo": "pdse",
        "url_origem": "https://editais.uff.br/sites/default/files/arquivos/Edital_02_PDSE_2026_.pdf",
    },
    "capes_22_2026.pdf": {
        "instituicao": "CAPES", "nivel": "pos", "tipo": "pdse",
        "url_origem": "https://www.gov.br/capes/pt-br/centrais-de-conteudo/editais/sei_2849220_edital_n__22_2026.pdf/@@display-file/file",
    },
    "usp_internacionalizacao.pdf": {
        "instituicao": "USP", "nivel": "graduacao", "tipo": "mobilidade",
        "url_origem": "https://prip.usp.br/wp-content/uploads/sites/1632/2026/08/Edital-2336-%E2%80%93-Internacionalizacao-com-Inclusao.pdf",
    },
}

"""Carrega cada PDF página a página e anexa os metadados do catálogo."""
def carregar_documentos(pasta=PASTA_DOCS, catalogo=CATALOGO):
    faltando = [n for n in catalogo if not (pasta / n).exists()]
    if faltando:
        raise FileNotFoundError(f"PDFs não encontrados: {faltando}")

    paginas = []
    for nome, meta in catalogo.items():
        for doc in PyMuPDFLoader(str(pasta / nome)).load():
            doc.metadata.update(meta)
            doc.metadata["fonte"] = nome
            paginas.append(doc)
    return paginas

"""Mostra uma tabela-resumo e uma prévia legível de cada documento."""
def exibir_resumo(paginas):
    linhas = []
    for nome, meta in CATALOGO.items():
        pgs = [p for p in paginas if p.metadata["fonte"] == nome]
        linhas.append({
            "Documento": nome,
            "Instituição": meta["instituicao"],
            "Nível": meta["nivel"],
            "Tipo": meta["tipo"],
            "Páginas": len(pgs),
            "Caracteres": sum(len(p.page_content) for p in pgs),
            "Págs. vazias": sum(1 for p in pgs if not p.page_content.strip()),
        })
    df = pd.DataFrame(linhas)
    display(df)
    print(f"\nTotal: {len(paginas)} páginas em {len(df)} documentos\n")

    for nome in CATALOGO:
        primeira = next(p for p in paginas if p.metadata["fonte"] == nome)
        texto = " ".join(primeira.page_content.split())
        print("=" * 90)
        print(f"{nome}  |  {CATALOGO[nome]['instituicao']}")
        print("-" * 90)
        print(textwrap.fill(texto[:400], width=90) + " ...\n")

paginas = carregar_documentos()
exibir_resumo(paginas)

,Documento,Instituição,Nível,Tipo,Páginas,Caracteres,Págs. vazias
0,unifesp_002_2026.pdf,UNIFESP,graduacao,mobilidade,11,21278,0
1,unifafibe_prri_01_2026.pdf,UNIFAFIBE,graduacao,mobilidade,10,17672,0
2,ufc_prointer_20_2026.pdf,UFC,graduacao,mobilidade,6,15284,0
3,uff_pdse_02_2026.pdf,UFF,pos,pdse,4,10599,0
4,capes_22_2026.pdf,CAPES,pos,pdse,15,50272,0
5,usp_internacionalizacao.pdf,USP,graduacao,mobilidade,4,17821,0



Total: 50 páginas em 6 documentos

unifesp_002_2026.pdf  |  UNIFESP
------------------------------------------------------------------------------------------
SERVIÇO PÚBLICO FEDERAL MINISTÉRIO DA EDUCAÇÃO UNIVERSIDADE FEDERAL DE SÃO PAULO
SECRETARIA DE RELAÇÕES INTERNACIONAIS EDITAL SRI/UNIFESP Nº 002/2026, DE 16 DE MARÇO DE
2026 EDITAL DE SELEÇÃO DE ESTUDANTES DE GRADUAÇÃO PARA MOBILIDADE ACADÊMICA EM
UNIVERSIDADES ESTRANGEIRAS CONVENIADAS, COM CONCESSÃO DE 2 (DUAS) BOLSAS-AUXÍLIO.
Estabelece normas gerais para seleção e indicação de alunos (as) da gr ...

unifafibe_prri_01_2026.pdf  |  UNIFAFIBE
------------------------------------------------------------------------------------------
EDITAL PRRI Nº 01/2026 PROCESSO SELETIVO DE ESTUDANTES DE GRADUAÇÃO PARA O PROGRAMA DE
MOBILIDADE INTERNACIONAL DO CENTRO UNIVERSITÁRIO UNIFAFIBE ANO LETIVO 2026–2027 O Pró-
Reitor Acadêmico do Centro Universitário UNIFAFIBE, Prof. Me. Rafael Catani Lima, e o
Coordenador Geral da Pós-Graduação, Pesqui

In [7]:
import statistics
from langchain_text_splitters import RecursiveCharacterTextSplitter

SEPARADORES = ["\n\n", "\n", ". ", " ", ""]
CONFIGS = [512, 1024]

"""Divide em chunks; overlap = 12,5% do chunk_size (dentro dos 10-15% pedidos)."""
def dividir(paginas, chunk_size):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_size // 8,
        separators=SEPARADORES,
    )
    return splitter.split_documents(paginas)

chunks_por_config = {cs: dividir(paginas, cs) for cs in CONFIGS}

resumo = []
for cs, chunks in chunks_por_config.items():
    tam = [len(c.page_content) for c in chunks]
    resumo.append({
        "chunk_size": cs,
        "overlap": cs // 8,
        "chunks": len(chunks),
        "média (chars)": round(statistics.mean(tam)),
        "mín": min(tam),
        "máx": max(tam),
    })
display(pd.DataFrame(resumo))

,chunk_size,overlap,chunks,média (chars),mín,máx
0,512,64,310,441,52,511
1,1024,128,166,858,155,1023


In [8]:
import shutil, time
from langchain_chroma import Chroma

"""Cria a coleção do domínio para um chunk_size, indexando em lotes."""
def criar_vectorstore(chunks, chunk_size, lote=32):
    pasta = f"/content/chroma_intercambio_{chunk_size}"
    shutil.rmtree(pasta, ignore_errors=True)
    db = Chroma(
        collection_name="intercambio_editais",
        collection_metadata={"hnsw:space": "cosine"},
        embedding_function=embeddings,
        persist_directory=pasta,
    )
    t0 = time.time()
    for i in range(0, len(chunks), lote):
        db.add_documents(chunks[i:i + lote])
        feitos = min(i + lote, len(chunks))
        print(f"  chunk_size={chunk_size}: {feitos}/{len(chunks)} "
              f"({time.time() - t0:.0f}s)", flush=True)
    return db

vectorstores = {}
for cs, chunks in chunks_por_config.items():
    vectorstores[cs] = criar_vectorstore(chunks, cs)

print()
for cs, db in vectorstores.items():
    print(f"chunk_size={cs}: {db._collection.count()} chunks indexados")

  chunk_size=512: 32/310 (31s)
  chunk_size=512: 64/310 (60s)
  chunk_size=512: 96/310 (94s)
  chunk_size=512: 128/310 (126s)
  chunk_size=512: 160/310 (162s)
  chunk_size=512: 192/310 (193s)
  chunk_size=512: 224/310 (226s)
  chunk_size=512: 256/310 (259s)
  chunk_size=512: 288/310 (291s)
  chunk_size=512: 310/310 (312s)
  chunk_size=1024: 32/166 (57s)
  chunk_size=1024: 64/166 (117s)
  chunk_size=1024: 96/166 (187s)
  chunk_size=1024: 128/166 (254s)
  chunk_size=1024: 160/166 (320s)
  chunk_size=1024: 166/166 (327s)

chunk_size=512: 310 chunks indexados
chunk_size=1024: 166 chunks indexados


In [9]:
"""Busca os k trechos mais relevantes nos editais."""
def buscar(consulta: str, k: int = 3, chunk_size: int = 512,
           instituicao: str = None, nivel: str = None, tipo: str = None) -> list[dict]:
    filtros = {"instituicao": instituicao, "nivel": nivel, "tipo": tipo}
    condicoes = [{campo: valor} for campo, valor in filtros.items() if valor]
    where = None
    if len(condicoes) == 1:
        where = condicoes[0]
    elif len(condicoes) > 1:
        where = {"$and": condicoes}

    resultados = vectorstores[chunk_size].similarity_search_with_score(
        consulta, k=k, filter=where
    )
    return [{
        "texto": doc.page_content,
        "fonte": doc.metadata["fonte"],
        "instituicao": doc.metadata["instituicao"],
        "pagina": doc.metadata["page"] + 1,
        "url_origem": doc.metadata["url_origem"],
        "similaridade": round(1 - dist, 3),
    } for doc, dist in resultados]

for r in buscar("prazo de inscrição", k=2, instituicao="UFC"):
    print(r["instituicao"], r["fonte"], "pág.", r["pagina"], r["similaridade"])

UFC ufc_prointer_20_2026.pdf pág. 2 0.709
UFC ufc_prointer_20_2026.pdf pág. 2 0.69


In [10]:
import re

INSTITUICOES = ["UNIFESP", "UNIFAFIBE", "UFC", "UFF", "CAPES", "USP"]

"""Extrai filtros de metadados citados na própria pergunta."""
def detectar_filtros(pergunta: str) -> dict:
    filtros = {}
    citadas = [i for i in INSTITUICOES
               if re.search(rf"\b{i}\b", pergunta, flags=re.IGNORECASE)]
    if len(citadas) == 1:
        filtros["instituicao"] = citadas[0]
    if re.search(r"\bPDSE\b", pergunta, flags=re.IGNORECASE):
        filtros["tipo"] = "pdse"
    return filtros

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

PROMPT_RAG = """<persona>
Você é um assistente especializado em editais de intercâmbio acadêmico.
Responda com base EXCLUSIVAMENTE nos trechos de editais fornecidos.
</persona>

<instrucoes>
- Responda SOMENTE com informações do contexto abaixo.
- Cite a fonte de cada informação no formato (fonte: nome_do_arquivo, página X).
- Se a resposta não estiver no contexto, diga exatamente:
  "Não encontrei essa informação nos documentos fornecidos."
- Nunca invente, extrapole ou use conhecimento externo.
- Responda em português do Brasil, de forma objetiva.
</instrucoes>

<contexto>
{contexto}
</contexto>

<pergunta>
{pergunta}
</pergunta>"""

chain_geracao = ChatPromptTemplate.from_template(PROMPT_RAG) | llm | StrOutputParser()

"""Formata os trechos recuperados com fonte e página para citação."""
def formatar_contexto(trechos: list[dict]) -> str:
    return "\n\n---\n\n".join(
        f"[Trecho {i} | fonte: {t['fonte']}, página {t['pagina']}]\n{t['texto']}"
        for i, t in enumerate(trechos, 1)
    )

"""Pipeline completo: retrieve -> generate. filtrar=False desliga o filtro automático."""
def responder(pergunta: str, k: int = 3, chunk_size: int = 512,
              filtrar: bool = True, **filtros) -> dict:
    if filtrar:
        filtros = {**detectar_filtros(pergunta), **filtros}
    trechos = buscar(pergunta, k=k, chunk_size=chunk_size, **filtros)
    resposta = chain_geracao.invoke({
        "contexto": formatar_contexto(trechos),
        "pergunta": pergunta,
    })
    return {"pergunta": pergunta, "resposta": resposta, "trechos": trechos}

"""Exibe resposta e fontes de forma legível."""
def mostrar(resultado: dict):
    print("=" * 90)
    print("PERGUNTA:", resultado["pergunta"])
    print("-" * 90)
    for linha in resultado["resposta"].splitlines():
        print(textwrap.fill(linha, width=90) if linha.strip() else "")
    print("\nFontes recuperadas:")
    for t in resultado["trechos"]:
        print(f"  - {t['fonte']} | pág. {t['pagina']} | similaridade {t['similaridade']}")
    print()

for p in [
    "Quais são os requisitos para se candidatar ao edital da UFC?",
    "Qual é o prazo de duração da bolsa do PDSE da CAPES?",
    "Qual é a previsão do tempo em Lisboa?",
]:
    mostrar(responder(p))

PERGUNTA: Quais são os requisitos para se candidatar ao edital da UFC?
------------------------------------------------------------------------------------------
Os requisitos mencionados são:
- capacidade de reflexão crítica sobre a experiência internacional e seus impactos
acadêmicos, pessoais e institucionais (fonte: ufc_prointer_20_2026.pdf, página 4);
- disponibilidade, maturidade e comprometimento para cumprir integralmente o período de
mobilidade, bem como para retornar à UFC após a finalização do curso pretendido (fonte:
ufc_prointer_20_2026.pdf, página 4);
- potencial de multiplicação dos conhecimentos adquiridos no retorno à UFC (fonte:
ufc_prointer_20_2026.pdf, página 4).

Fontes recuperadas:
  - ufc_prointer_20_2026.pdf | pág. 5 | similaridade 0.746
  - ufc_prointer_20_2026.pdf | pág. 4 | similaridade 0.741
  - ufc_prointer_20_2026.pdf | pág. 6 | similaridade 0.734

PERGUNTA: Qual é o prazo de duração da bolsa do PDSE da CAPES?
----------------------------------------------

In [12]:
import time

PERGUNTAS = [
    "Quais documentos o candidato precisa apresentar na inscrição do edital da UNIFESP?",
    "Qual é o prazo de duração da bolsa do PDSE da CAPES?",
    "Quem pode se candidatar ao edital de intercâmbio do UNIFAFIBE?",
    "Quais são os critérios de seleção do edital da UFC?",
    "Quais são as condições para acumular o PDSE com outra bolsa?",
    "O que a USP exige de quem foi selecionado para o intercâmbio?",
]

"""Roda o pipeline RAG para cada pergunta e guarda resposta + contextos."""
def coletar(chunk_size, perguntas=PERGUNTAS, k=3, filtrar=False):
    dados = []
    for q in perguntas:
        r = responder(q, k=k, chunk_size=chunk_size, filtrar=filtrar)
        dados.append({
            "pergunta": q,
            "resposta": r["resposta"],
            "contextos": [t["texto"] for t in r["trechos"]],
        })
    return dados

t0 = time.time()
coleta = {}
for cs in CONFIGS:
    coleta[cs] = coletar(cs)
    print(f"chunk_size={cs}: {len(coleta[cs])} perguntas respondidas ({time.time() - t0:.0f}s)")

chunk_size=512: 6 perguntas respondidas (6s)
chunk_size=1024: 6 perguntas respondidas (13s)


In [13]:
from ragas import evaluate, EvaluationDataset
from ragas.run_config import RunConfig
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(llm)
ragas_embs = LangchainEmbeddingsWrapper(embeddings)

"""Calcula faithfulness e answer_relevancy por pergunta para um chunk_size."""
def avaliar(chunk_size):
    dataset = EvaluationDataset.from_list([
        {"user_input": d["pergunta"],
         "response": d["resposta"],
         "retrieved_contexts": d["contextos"]}
        for d in coleta[chunk_size]
    ])
    res = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy],
        llm=ragas_llm,
        embeddings=ragas_embs,
        run_config=RunConfig(timeout=180, max_workers=4),
        raise_exceptions=False,
    )
    df = res.to_pandas()
    return pd.DataFrame({
        "chunk_size": chunk_size,
        "pergunta": [d["pergunta"] for d in coleta[chunk_size]],
        "faithfulness": df["faithfulness"].round(3),
        "answer_relevancy": df["answer_relevancy"].round(3),
    })

tabelas = {}
for cs in CONFIGS:
    print(f"Avaliando chunk_size={cs} ...")
    tabelas[cs] = avaliar(cs)
    display(tabelas[cs])

comparacao = pd.DataFrame([{
    "chunk_size": cs,
    "faithfulness (média)": round(t["faithfulness"].mean(), 3),
    "answer_relevancy (média)": round(t["answer_relevancy"].mean(), 3),
    "perguntas sem nota": int(t[["faithfulness", "answer_relevancy"]].isna().any(axis=1).sum()),
} for cs, t in tabelas.items()])
display(comparacao)

Avaliando chunk_size=512 ...


/tmp/ipykernel_5650/4062871500.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
/tmp/ipykernel_5650/4062871500.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy
/tmp/ipykernel_5650/4062871500.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)
/tmp/ipykernel_5650/4062871500.py:8: Deprecation

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

,chunk_size,pergunta,faithfulness,answer_relevancy
0,512,Quais documentos o candidato precisa apresenta...,0.000,0.000
1,512,Qual é o prazo de duração da bolsa do PDSE da ...,0.500,0.848
2,512,Quem pode se candidatar ao edital de intercâmb...,1.000,0.000
3,512,Quais são os critérios de seleção do edital da...,1.000,0.831
4,512,Quais são as condições para acumular o PDSE co...,1.000,0.928
5,512,O que a USP exige de quem foi selecionado para...,0.667,0.976


Avaliando chunk_size=1024 ...


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

,chunk_size,pergunta,faithfulness,answer_relevancy
0,1024,Quais documentos o candidato precisa apresenta...,1.0,0.000
1,1024,Qual é o prazo de duração da bolsa do PDSE da ...,1.0,0.000
2,1024,Quem pode se candidatar ao edital de intercâmb...,1.0,0.000
3,1024,Quais são os critérios de seleção do edital da...,1.0,0.831
4,1024,Quais são as condições para acumular o PDSE co...,1.0,0.862
5,1024,O que a USP exige de quem foi selecionado para...,1.0,0.654


,chunk_size,faithfulness (média),answer_relevancy (média),perguntas sem nota
0,512,0.694,0.597,0
1,1024,1.000,0.391,0


In [14]:
print("Perguntas com answer_relevancy = 0 (resposta + fontes recuperadas)\n")
for cs in CONFIGS:
    for i, linha in tabelas[cs].reset_index(drop=True).iterrows():
        if linha["answer_relevancy"] == 0:
            item = coleta[cs][i]
            print("=" * 90)
            print(f"chunk_size={cs} | {item['pergunta']}")
            print("-" * 90)
            print(textwrap.fill(" ".join(item["resposta"].split())[:500], width=90))
            print("\nFontes recuperadas:")
            for t in buscar(item["pergunta"], k=3, chunk_size=cs):
                print(f"  - {t['fonte']} | pág. {t['pagina']} | sim. {t['similaridade']}")
            print()

Perguntas com answer_relevancy = 0 (resposta + fontes recuperadas)

chunk_size=512 | Quais documentos o candidato precisa apresentar na inscrição do edital da UNIFESP?
------------------------------------------------------------------------------------------
Não encontrei essa informação nos documentos fornecidos.

Fontes recuperadas:
  - usp_internacionalizacao.pdf | pág. 2 | sim. 0.787
  - ufc_prointer_20_2026.pdf | pág. 2 | sim. 0.779
  - uff_pdse_02_2026.pdf | pág. 1 | sim. 0.777

chunk_size=512 | Quem pode se candidatar ao edital de intercâmbio do UNIFAFIBE?
------------------------------------------------------------------------------------------
Não encontrei essa informação nos documentos fornecidos.

Fontes recuperadas:
  - usp_internacionalizacao.pdf | pág. 2 | sim. 0.749
  - capes_22_2026.pdf | pág. 15 | sim. 0.724
  - ufc_prointer_20_2026.pdf | pág. 4 | sim. 0.722

chunk_size=1024 | Quais documentos o candidato precisa apresentar na inscrição do edital da UNIFESP?
---------

In [15]:
""" Guarda o baseline (respostas SEM filtro) uma única vez """
if "baseline" not in globals():
    baseline = {"coleta": coleta, "tabelas": tabelas}

""" Coleta de novo, agora com filtro automático de metadados """
coleta = {cs: coletar(cs, filtrar=True) for cs in CONFIGS}

tabelas = {}
for cs in CONFIGS:
    print(f"Avaliando chunk_size={cs} ...")
    tabelas[cs] = avaliar(cs)

def medias(tabs, rotulo):
    return [{
        "versão": rotulo, "chunk_size": cs,
        "faithfulness (média)": round(t["faithfulness"].mean(), 3),
        "answer_relevancy (média)": round(t["answer_relevancy"].mean(), 3),
        "notas zero (relevancy)": int((t["answer_relevancy"] == 0).sum()),
    } for cs, t in tabs.items()]

display(pd.DataFrame(medias(baseline["tabelas"], "sem filtro")
                     + medias(tabelas, "com filtro")))

Avaliando chunk_size=512 ...


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

Avaliando chunk_size=1024 ...


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

,versão,chunk_size,faithfulness (média),answer_relevancy (média),notas zero (relevancy)
0,sem filtro,512,0.694,0.597,2
1,sem filtro,1024,1.000,0.391,3
2,com filtro,512,0.792,0.856,0
3,com filtro,1024,0.944,0.422,3


In [16]:
q = PERGUNTAS[0]
filtros = detectar_filtros(q)

for cs in CONFIGS:
    print("=" * 90)
    print(f"chunk_size={cs} | filtros={filtros}")
    for t in buscar(q, k=3, chunk_size=cs, **filtros):
        print(f"\n[pág. {t['pagina']} | sim. {t['similaridade']}]")
        print(textwrap.fill(" ".join(t["texto"].split()), width=90))

print("\n" + "=" * 90)
for p in paginas:
    if p.metadata["fonte"] == "unifesp_002_2026.pdf":
        n = len(re.findall(r"document", p.page_content, flags=re.IGNORECASE))
        print(f"pág. {p.metadata['page'] + 1}: {n} menções")

chunk_size=512 | filtros={'instituicao': 'UNIFESP'}

[pág. 7 | sim. 0.768]
situações, tais documentos deverão ser assinados pelo(a) Coordenador(a) do Curso e pelo(a)
Coordenador(a) Institucional da Unifesp, representado(a) pela Secretaria de Relações
Internacionais.

[pág. 4 | sim. 0.767]
rigorosamente na ordem em que foram apresentados neste edital. No campo assunto do e-mail
deverá constar: Inscrição – Edital Conjunto 2026. 3.4. Não serão aceitos documentos
enviados por fax, correio ou após a data e o horário estabelecidos na cláusula 3.2. 3.5. A
ausência de documentos obrigatórios ou o envio de formulários incompletos ou não
preenchidos implicará a desclassificação da candidatura. 4. DAS INSTITUIÇÕES E DAS VAGAS:
Instituição e País Vagas Área Página Web da Instituição:

[pág. 3 | sim. 0.753]
poder público; 3. DAS INSCRIÇÕES: 3.1. O (A) candidato (a) interessado (a) em participar
do processo seletivo deverá apresentar à Secretaria de Relações Internacionais da Unifesp
a documentação 

## Análise de falha: pergunta sobre documentos da inscrição (UNIFESP)

**Pergunta:** "Quais documentos o candidato precisa apresentar na inscrição do edital da UNIFESP?"

**O que aconteceu:**
- Sem filtro de metadados, o retriever trouxe trechos de outros editais (USP, UFC, UFF) e o modelo respondeu "Não encontrei".
- Com filtro por instituição, os trechos passaram a ser da UNIFESP.
- Com chunk_size=512, o trecho correto (item 3.1, página 3, lista de documentos) foi recuperado e a pergunta foi respondida.
- Com chunk_size=1024, o trecho correto não ficou entre os 3 melhores, e a resposta continuou "Não encontrei".

**Hipótese:** no chunk maior, a lista de documentos fica misturada com outros assuntos do edital, o que dilui a similaridade com a pergunta.

**Limitação do 512:** o chunk termina no item III da lista, então a resposta pode ficar incompleta.

**Conclusão:** chunk_size=512 com filtro de metadados foi a melhor configuração neste corpus.

In [17]:
from sentence_transformers import CrossEncoder

MODELO_RERANK = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(MODELO_RERANK)
print("Reranker carregado:", MODELO_RERANK)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker carregado: cross-encoder/ms-marco-MiniLM-L-6-v2


In [18]:
"""Busca k_inicial trechos, reordena com o cross-encoder e devolve os k melhores."""
def buscar_rerank(consulta: str, k: int = 3, k_inicial: int = 10,
                  chunk_size: int = 512, **filtros) -> list[dict]:
    candidatos = buscar(consulta, k=k_inicial, chunk_size=chunk_size, **filtros)
    notas = reranker.predict([(consulta, c["texto"]) for c in candidatos])
    for c, nota in zip(candidatos, notas):
        c["score_rerank"] = round(float(nota), 3)
    candidatos.sort(key=lambda c: c["score_rerank"], reverse=True)
    return candidatos[:k]

In [19]:
"""retrieve (filtro + rerank) -> generate."""
def responder_rerank(pergunta: str, k: int = 3, chunk_size: int = 512, **filtros) -> dict:
    filtros = {**detectar_filtros(pergunta), **filtros}
    trechos = buscar_rerank(pergunta, k=k, chunk_size=chunk_size, **filtros)
    resposta = chain_geracao.invoke({
        "contexto": formatar_contexto(trechos),
        "pergunta": pergunta,
    })
    return {"pergunta": pergunta, "resposta": resposta, "trechos": trechos}

"""RAGAS por pergunta para uma lista de {pergunta, resposta, contextos}."""
def avaliar_dados(dados: list[dict], chunk_size: int) -> pd.DataFrame:
    dataset = EvaluationDataset.from_list([
        {"user_input": d["pergunta"], "response": d["resposta"],
         "retrieved_contexts": d["contextos"]} for d in dados
    ])
    res = evaluate(
        dataset, metrics=[faithfulness, answer_relevancy],
        llm=ragas_llm, embeddings=ragas_embs,
        run_config=RunConfig(timeout=180, max_workers=4),
        raise_exceptions=False,
    )
    df = res.to_pandas()
    return pd.DataFrame({
        "chunk_size": chunk_size,
        "pergunta": [d["pergunta"] for d in dados],
        "faithfulness": df["faithfulness"].round(3),
        "answer_relevancy": df["answer_relevancy"].round(3),
    })

if "tabelas_filtro" not in globals():
    tabelas_filtro = tabelas

coleta_rerank, tabelas_rerank = {}, {}
for cs in CONFIGS:
    print(f"Coletando e avaliando chunk_size={cs} com rerank ...")
    coleta_rerank[cs] = []
    for q in PERGUNTAS:
        r = responder_rerank(q, chunk_size=cs)
        coleta_rerank[cs].append({
            "pergunta": q,
            "resposta": r["resposta"],
            "contextos": [t["texto"] for t in r["trechos"]],
        })
    tabelas_rerank[cs] = avaliar_dados(coleta_rerank[cs], cs)
    display(tabelas_rerank[cs])

display(pd.DataFrame(
    medias(baseline["tabelas"], "sem filtro")
    + medias(tabelas_filtro, "com filtro")
    + medias(tabelas_rerank, "filtro + rerank")
))

Coletando e avaliando chunk_size=512 com rerank ...


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

,chunk_size,pergunta,faithfulness,answer_relevancy
0,512,Quais documentos o candidato precisa apresenta...,0.857,0.858
1,512,Qual é o prazo de duração da bolsa do PDSE da ...,0.667,0.768
2,512,Quem pode se candidatar ao edital de intercâmb...,0.500,0.734
3,512,Quais são os critérios de seleção do edital da...,1.000,0.610
4,512,Quais são as condições para acumular o PDSE co...,1.000,0.928
5,512,O que a USP exige de quem foi selecionado para...,1.000,0.827


Coletando e avaliando chunk_size=1024 com rerank ...


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

,chunk_size,pergunta,faithfulness,answer_relevancy
0,1024,Quais documentos o candidato precisa apresenta...,1.000,0.000
1,1024,Qual é o prazo de duração da bolsa do PDSE da ...,1.000,0.000
2,1024,Quem pode se candidatar ao edital de intercâmb...,1.000,0.000
3,1024,Quais são os critérios de seleção do edital da...,0.714,0.663
4,1024,Quais são as condições para acumular o PDSE co...,1.000,0.869
5,1024,O que a USP exige de quem foi selecionado para...,1.000,0.609


,versão,chunk_size,faithfulness (média),answer_relevancy (média),notas zero (relevancy)
0,sem filtro,512,0.694,0.597,2
1,sem filtro,1024,1.000,0.391,3
2,com filtro,512,0.792,0.856,0
3,com filtro,1024,0.944,0.422,3
4,filtro + rerank,512,0.837,0.788,0
5,filtro + rerank,1024,0.952,0.357,3


## Reranking (diferencial)

Cross-encoder `cross-encoder/ms-marco-MiniLM-L-6-v2`: o retriever busca 10 trechos, o
cross-encoder reordena e os 3 melhores seguem para o LLM.

No RAGAS, as médias com rerank ficaram equivalentes às da versão só com filtro
(diferenças dentro da variação do juiz, com apenas 6 perguntas). O modelo é treinado em
inglês, o que pode limitar o ganho em português.

## Comparação final e escolha da configuração

Resultados da execução final (6 perguntas, RAGAS com `gemma4:cloud` como juiz):

| Versão | chunk_size | faithfulness (média) | answer_relevancy (média) | notas zero |
|---|---|---|---|---|
| sem filtro | 512 | 0,694 | 0,597 | 2 |
| sem filtro | 1024 | 1,000 | 0,391 | 3 |
| com filtro | 512 | 0,792 | 0,856 | 0 |
| com filtro | 1024 | 0,944 | 0,422 | 3 |
| filtro + rerank | 512 | 0,837 | 0,788 | 0 |
| filtro + rerank | 1024 | 0,952 | 0,357 | 3 |

**Escolha: `chunk_size=512` com filtro de metadados.**

- É a configuração com menos respostas "Não encontrei" (0 notas zero contra 3 no 1024) e,
  com o filtro, o faithfulness médio fica acima da meta de 0,7 (0,792).
- O filtro de metadados foi a melhoria de maior efeito no 512: o faithfulness passou de
  0,694 para 0,792 e as notas zero caíram de 2 para 0.
- A faithfulness alta do 1024 vem de respostas que não afirmam nada, porque o trecho com a
  resposta não foi recuperado (ver análise de falha).
- O reranking deu resultados equivalentes aos do filtro sozinho no 512 (faithfulness
  sobe, relevancy desce, dentro da variação entre execuções).
- Limitação: só 6 perguntas e o juiz (LLM) varia entre execuções, então diferenças
  pequenas não são conclusivas.